# 02 Image Captioning

- Implement image captioning
- Uses BLIP to generate captions from input images

## Setup

In [ ]:
# Install required libraries
!pip install pandas transformers gTTS torch torchvision pillow

  Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached gTTS-2.5.4-py3-none-any.whl.metadata (4.1 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp314-cp314-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
INFO: pip is looking at multiple versions of typer to determine which version is compatib

## Generate captions

In [1]:
import torch
from pathlib import Path
from PIL import Image
import pandas as pd
from transformers import BlipProcessor, BlipForConditionalGeneration

# Paths
images_dir = Path("../data/coco_test")
output_dir = Path("../results/captions")
output_dir.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load processor and model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# Load image files
image_files = sorted([f for f in images_dir.iterdir() if f.suffix.lower() in ['.jpg']])

# Generate caption for an image
def generate_caption(image_path, max_new_tokens=20):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    caption = processor.decode(outputs[0], skip_special_tokens=True)
    return caption

results = []
for image_file in image_files:
    try:
        caption = generate_caption(image_file)
        results.append({
            "image": image_file.name,
            "caption": caption
        })
        print(f"{image_file.name}: {caption}")
    except Exception as e:
        results.append({
            "image": image_file.name,
            "caption": None,
            "error": str(e)
        })

captions_df = pd.DataFrame(results)
captions_df.to_csv(output_dir / "blip_captions.csv", index=False)

captions_df.head()

/Users/manumalakannavar/Documents/COM3025-CW/COM3025_Virtual_Env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 66015.77it/s]


000000000139.jpg: a living room with a television and a fireplace
000000000285.jpg: a brown bear sitting in the grass
000000000632.jpg: a bedroom with a bed and a window
000000000724.jpg: a stop sign on a street corner
000000000776.jpg: a brown teddy bear
000000000785.jpg: a woman in a red jacket skiing down a hill
000000000802.jpg: a kitchen with a white refrigerator and a white stove
000000000872.jpg: a man in a baseball uniform
000000000885.jpg: a man playing tennis
000000001000.jpg: a group of kids
000000001268.jpg: a woman taking a picture of a swan
000000001296.jpg: a woman holding a cell phone
000000001353.jpg: a group of people riding on a red train
000000001425.jpg: a plate with a sandwich on it
000000001490.jpg: a man on a surfboard
000000001503.jpg: a white desk
000000001532.jpg: a freeway with a bunch of cars driving underneath
000000001584.jpg: a red double decker bus driving down a street
000000001675.jpg: a black and white cat sitting on a computer
000000001761.jpg: a br

,image,caption
0,000000000139.jpg,a living room with a television and a fireplace
1,000000000285.jpg,a brown bear sitting in the grass
2,000000000632.jpg,a bedroom with a bed and a window
3,000000000724.jpg,a stop sign on a street corner
4,000000000776.jpg,a brown teddy bear


## Generate audio files

In [2]:
from pathlib import Path
from gtts import gTTS

# Audio Path
audio_dir = Path("../results/audio")
audio_dir.mkdir(parents=True, exist_ok=True)

# Convert captions to audio
def caption_to_audio(caption, output_path):
    tts = gTTS(text=caption, lang='en', slow=False)
    tts.save(output_path)

audio_results = []
for idx, row in captions_df.iterrows():
    image = row['image']
    caption = row['caption']

    # Skip if caption is missing
    if pd.isna(caption):
        continue

    audio_file_name = Path(image).stem + ".mp3"
    audio_path = audio_dir / audio_file_name

    try:
        caption_to_audio(caption, audio_path)

        audio_results.append({
            "image": image,
            "caption": caption,
            "audio": audio_file_name
        })

        print(f"Created audio for {image}: {audio_file_name}")
    except Exception as e:
        audio_results.append({
            "image": image,
            "caption": caption,
            "audio": None,
            "error": str(e)
        })

audio_df = pd.DataFrame(audio_results)
audio_df.to_csv(output_dir / "audio_results.csv", index=False)
audio_df.head()

Created audio for 000000000139.jpg: 000000000139.mp3
Created audio for 000000000285.jpg: 000000000285.mp3
Created audio for 000000000632.jpg: 000000000632.mp3
Created audio for 000000000724.jpg: 000000000724.mp3
Created audio for 000000000776.jpg: 000000000776.mp3
Created audio for 000000000785.jpg: 000000000785.mp3
Created audio for 000000000802.jpg: 000000000802.mp3
Created audio for 000000000872.jpg: 000000000872.mp3
Created audio for 000000000885.jpg: 000000000885.mp3
Created audio for 000000001000.jpg: 000000001000.mp3
Created audio for 000000001268.jpg: 000000001268.mp3
Created audio for 000000001296.jpg: 000000001296.mp3
Created audio for 000000001353.jpg: 000000001353.mp3
Created audio for 000000001425.jpg: 000000001425.mp3
Created audio for 000000001490.jpg: 000000001490.mp3
Created audio for 000000001503.jpg: 000000001503.mp3
Created audio for 000000001532.jpg: 000000001532.mp3
Created audio for 000000001584.jpg: 000000001584.mp3
Created audio for 000000001675.jpg: 0000000016

,image,caption,audio
0,000000000139.jpg,a living room with a television and a fireplace,000000000139.mp3
1,000000000285.jpg,a brown bear sitting in the grass,000000000285.mp3
2,000000000632.jpg,a bedroom with a bed and a window,000000000632.mp3
3,000000000724.jpg,a stop sign on a street corner,000000000724.mp3
4,000000000776.jpg,a brown teddy bear,000000000776.mp3
